# Smart Desk Pet — Model Compression Comparison
**EE 446 Tiny Machine Learning**  
Jocelyn He & Pandha Panthawangkun

This notebook compares four model configurations:
1. Baseline (float32)
2. Post-Training Quantization (PTQ int8)
3. Quantization-Aware Training (QAT)
4. Pruning + PTQ


## Step 1 — Install dependencies

In [ ]:
!pip install tensorflow tensorflow-model-optimization numpy pandas scikit-learn cbor2 -q

## Step 2 — Upload your CBOR files from Edge Impulse

**How to export from Edge Impulse (terminal):**
```
edge-impulse-data-export --out-dir ~/Desktop/ei-data
```
This creates a folder like:
```
ei-data/
  training/
    good.1234.cbor
    slouch.5678.cbor
  testing/
    good.9012.cbor
    slouch.3456.cbor
```
Upload ALL the `.cbor` files from both `training/` and `testing/` folders below.
The label is read from inside the CBOR file itself — no need to rename anything.

In [ ]:
from google.colab import files
import io, os
import cbor2
import pandas as pd
import numpy as np

uploaded = files.upload()
print(f'Uploaded {len(uploaded)} files:')
for name in uploaded:
    print(f'  {name}')

## Step 3 — Load and parse CBOR data
Reads each CBOR file, extracts the label and raw IMU values, applies sliding window feature extraction.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

WINDOW_SIZE = 50   # 1 second at 50 Hz
STEP_SIZE   = 5    # 100ms step

def extract_features_from_array(data_2d):
    """Extract statistical features from a (window x axes) numpy array."""
    features = []
    for col_idx in range(data_2d.shape[1]):
        vals = data_2d[:, col_idx].astype(float)
        features += [
            np.mean(vals),
            np.std(vals),
            np.min(vals),
            np.max(vals),
            np.sqrt(np.mean(vals**2)),            # RMS
            np.sum(np.abs(np.diff(vals)) > 0.1),  # zero-crossing proxy
        ]
    return features

def parse_cbor_file(content):
    """
    Parse an Edge Impulse CBOR file.
    Returns (label, numpy array of shape [n_samples, n_axes]).
    Edge Impulse CBOR structure:
      {
        'label': 'good',
        'payload': {
          'sensors': [{'name': 'accX', ...}, ...],
          'values': [[ax, ay, az, gx, gy, gz], ...]
        }
      }
    """
    obj = cbor2.loads(content)

    # Extract label — stored at top level or inside payload
    label_raw = obj.get('label', obj.get('payload', {}).get('label', ''))
    if not label_raw:
        # Try to get from 'name' field
        label_raw = obj.get('name', '')
    label_raw = str(label_raw).lower().strip()

    # Normalise label
    if any(x in label_raw for x in ['good', 'normal', 'upright']):
        label = 'good'
    elif any(x in label_raw for x in ['slouch', 'bad', 'slump']):
        label = 'slouch'
    else:
        label = label_raw  # keep as-is, will be reported as unknown

    # Extract sensor values
    payload = obj.get('payload', obj)  # some files put values at top level
    values  = payload.get('values', [])

    if not values:
        return label, None

    # values is a flat list or list-of-lists
    arr = np.array(values, dtype=np.float32)
    if arr.ndim == 1:
        # flat list — figure out n_axes from sensors metadata
        sensors = payload.get('sensors', [])
        n_axes  = len(sensors) if sensors else 6
        arr = arr.reshape(-1, n_axes)

    return label, arr

# Build dataset
X_list, y_list = [], []
label_counts = {}
skipped = []

for fname, content in uploaded.items():
    if not fname.endswith('.cbor'):
        print(f'  Skipping non-CBOR file: {fname}')
        skipped.append(fname)
        continue

    try:
        label, arr = parse_cbor_file(content)
    except Exception as e:
        print(f'  ERROR parsing {fname}: {e}')
        skipped.append(fname)
        continue

    if arr is None or len(arr) == 0:
        print(f'  WARNING: {fname} had no sensor values — skipping')
        skipped.append(fname)
        continue

    if label not in ['good', 'slouch']:
        print(f'  WARNING: {fname} has unrecognised label "{label}" — skipping')
        skipped.append(fname)
        continue

    # Sliding window feature extraction
    windows_this_file = 0
    for start in range(0, len(arr) - WINDOW_SIZE, STEP_SIZE):
        window = arr[start:start + WINDOW_SIZE]
        feats  = extract_features_from_array(window)
        if feats:
            X_list.append(feats)
            y_list.append(label)
            windows_this_file += 1

    label_counts[label] = label_counts.get(label, 0) + windows_this_file
    print(f'  {fname} -> label={label}, samples={len(arr)}, windows={windows_this_file}')

if skipped:
    print(f'\nSkipped files: {skipped}')

if not X_list:
    raise ValueError('No data loaded! Check that your CBOR files contain good/slouch labels.')

X = np.array(X_list, dtype=np.float32)
print(f'\nDataset: {X.shape[0]} windows, {X.shape[1]} features each')
print(f'Class distribution: {label_counts}')

## Step 4 — Prepare train/test split

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_enc = le.fit_transform(y_list)  # good=0, slouch=1 (alphabetical)
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# One-hot encode for Keras
import tensorflow as tf
y_train_oh = tf.keras.utils.to_categorical(y_train, 2)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  2)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print(f'Features per sample: {X_train.shape[1]}')

## Step 5 — Helper functions

In [ ]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import tempfile, os, time

N_FEATURES = X_train.shape[1]

def build_model():
    """Small fully-connected network matching Edge Impulse architecture."""
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(20, activation='relu', input_shape=(N_FEATURES,)),
        tf.keras.layers.Dense(20, activation='relu'),
        tf.keras.layers.Dense(2,  activation='softmax'),
    ], name='posture_classifier')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, epochs=50, verbose=0):
    model.fit(
        X_train, y_train_oh,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        verbose=verbose
    )
    return model

def evaluate_keras(model, label=''):
    _, acc = model.evaluate(X_test, y_test_oh, verbose=0)
    return round(acc * 100, 2)

def get_tflite_size_kb(tflite_model):
    return round(len(tflite_model) / 1024, 1)

def evaluate_tflite(tflite_model):
    """Run TFLite model on test set and return accuracy."""
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()

    correct = 0
    for i in range(len(X_test)):
        x = X_test[i:i+1].astype(
            np.float32 if inp[0]['dtype'] == np.float32 else np.int8
        )
        if inp[0]['dtype'] == np.int8:
            scale, zero_point = inp[0]['quantization']
            x = (X_test[i:i+1] / scale + zero_point).astype(np.int8)
        interpreter.set_tensor(inp[0]['index'], x)
        interpreter.invoke()
        pred = np.argmax(interpreter.get_tensor(out[0]['index']))
        if pred == y_test[i]:
            correct += 1
    return round(correct / len(X_test) * 100, 2)

def measure_tflite_latency(tflite_model, n_runs=100):
    """Measure average inference latency in ms."""
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    x = X_test[0:1].astype(np.float32)
    if inp[0]['dtype'] == np.int8:
        scale, zero_point = inp[0]['quantization']
        x = (x / scale + zero_point).astype(np.int8)
    start = time.perf_counter()
    for _ in range(n_runs):
        interpreter.set_tensor(inp[0]['index'], x)
        interpreter.invoke()
    elapsed = (time.perf_counter() - start) / n_runs * 1000
    return round(elapsed, 3)

def save_tflite(tflite_model, path):
    with open(path, 'wb') as f:
        f.write(tflite_model)

print('Helper functions ready.')

## Step 6 — Config 1: Baseline float32

In [ ]:
print('Training baseline float32 model...')
baseline_model = build_model()
baseline_model = train_model(baseline_model, epochs=50)
baseline_acc_keras = evaluate_keras(baseline_model)
print(f'Keras accuracy: {baseline_acc_keras}%')

# Convert to TFLite float32
converter = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
tflite_float32 = converter.convert()
save_tflite(tflite_float32, '/content/model_float32.tflite')

float32_size    = get_tflite_size_kb(tflite_float32)
float32_acc     = evaluate_tflite(tflite_float32)
float32_latency = measure_tflite_latency(tflite_float32)

print(f'\nFloat32 results:')
print(f'  Size:     {float32_size} KB')
print(f'  Accuracy: {float32_acc}%')
print(f'  Latency:  {float32_latency} ms (on Colab CPU)')

## Step 7 — Config 2: Post-Training Quantization (PTQ int8)

In [ ]:
print('Applying post-training quantization (int8)...')

def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter_ptq = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
converter_ptq.optimizations = [tf.lite.Optimize.DEFAULT]
converter_ptq.representative_dataset = representative_dataset
converter_ptq.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_ptq.inference_input_type  = tf.int8
converter_ptq.inference_output_type = tf.int8
tflite_ptq = converter_ptq.convert()
save_tflite(tflite_ptq, '/content/model_ptq_int8.tflite')

ptq_size    = get_tflite_size_kb(tflite_ptq)
ptq_acc     = evaluate_tflite(tflite_ptq)
ptq_latency = measure_tflite_latency(tflite_ptq)

print(f'\nPTQ int8 results:')
print(f'  Size:     {ptq_size} KB')
print(f'  Accuracy: {ptq_acc}%')
print(f'  Latency:  {ptq_latency} ms (on Colab CPU)')

## Step 8 — Config 3: Quantization-Aware Training (QAT)

In [ ]:
print('Training with quantization-aware training (QAT)...')

# Start from the trained baseline and apply QAT
qat_model = tfmot.quantization.keras.quantize_model(baseline_model)
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tune with quantization simulation
qat_model.fit(
    X_train, y_train_oh,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Convert QAT model to int8 TFLite
converter_qat = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter_qat.optimizations = [tf.lite.Optimize.DEFAULT]
converter_qat.representative_dataset = representative_dataset
converter_qat.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_qat.inference_input_type  = tf.int8
converter_qat.inference_output_type = tf.int8
tflite_qat = converter_qat.convert()
save_tflite(tflite_qat, '/content/model_qat_int8.tflite')

qat_size    = get_tflite_size_kb(tflite_qat)
qat_acc     = evaluate_tflite(tflite_qat)
qat_latency = measure_tflite_latency(tflite_qat)

print(f'\nQAT int8 results:')
print(f'  Size:     {qat_size} KB')
print(f'  Accuracy: {qat_acc}%')
print(f'  Latency:  {qat_latency} ms (on Colab CPU)')

## Step 9 — Config 4: Pruning + PTQ

In [ ]:
print('Training with pruning (50% sparsity) + PTQ...')

# Build a fresh model for pruning
pruning_model = build_model()

end_step = np.ceil(len(X_train) / 32).astype(np.int32) * 50

pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=end_step
    )
}

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(pruning_model, **pruning_params)
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_model.fit(
    X_train, y_train_oh,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=0
)

# Strip pruning wrappers
pruned_stripped = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Apply PTQ on top of pruned model
converter_pruned = tf.lite.TFLiteConverter.from_keras_model(pruned_stripped)
converter_pruned.optimizations = [tf.lite.Optimize.DEFAULT]
converter_pruned.representative_dataset = representative_dataset
converter_pruned.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_pruned.inference_input_type  = tf.int8
converter_pruned.inference_output_type = tf.int8
tflite_pruned = converter_pruned.convert()
save_tflite(tflite_pruned, '/content/model_pruned_ptq.tflite')

pruned_size    = get_tflite_size_kb(tflite_pruned)
pruned_acc     = evaluate_tflite(tflite_pruned)
pruned_latency = measure_tflite_latency(tflite_pruned)

# Calculate actual sparsity
total_weights, zero_weights = 0, 0
for w in pruned_stripped.weights:
    vals = w.numpy().flatten()
    total_weights += len(vals)
    zero_weights  += np.sum(vals == 0)
actual_sparsity = round(zero_weights / total_weights * 100, 1)

print(f'\nPruning + PTQ results:')
print(f'  Size:          {pruned_size} KB')
print(f'  Accuracy:      {pruned_acc}%')
print(f'  Latency:       {pruned_latency} ms (on Colab CPU)')
print(f'  Actual sparsity: {actual_sparsity}%')

## Step 10 — Full comparison table

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Configuration':  ['Float32 (baseline)', 'PTQ int8', 'QAT int8', 'Pruning 50% + PTQ'],
    'Model size (KB)': [float32_size, ptq_size, qat_size, pruned_size],
    'Accuracy (%)':   [float32_acc,  ptq_acc,  qat_acc,  pruned_acc],
    'Latency (ms)':   [float32_latency, ptq_latency, qat_latency, pruned_latency],
    'Size reduction': [
        '—',
        f'{round((1 - ptq_size/float32_size)*100)}% smaller',
        f'{round((1 - qat_size/float32_size)*100)}% smaller',
        f'{round((1 - pruned_size/float32_size)*100)}% smaller',
    ],
    'Accuracy delta': [
        '—',
        f'{round(ptq_acc - float32_acc, 2):+}%',
        f'{round(qat_acc - float32_acc, 2):+}%',
        f'{round(pruned_acc - float32_acc, 2):+}%',
    ]
})

print('=' * 80)
print('SMART DESK PET — MODEL COMPRESSION COMPARISON')
print('=' * 80)
print(results.to_string(index=False))
print('=' * 80)
print(f'\nNote: Latency measured on Colab CPU, not Arduino Nano 33 BLE Sense.')
print(f'On-device latency from Edge Impulse: float32=20ms, PTQ int8=19ms')
print(f'Arduino Nano 33 BLE Sense RAM: 256KB total | Flash: 1MB total')

## Step 11 — Download all TFLite models

In [ ]:
from google.colab import files

print('Downloading all TFLite models...')
for path in [
    '/content/model_float32.tflite',
    '/content/model_ptq_int8.tflite',
    '/content/model_qat_int8.tflite',
    '/content/model_pruned_ptq.tflite'
]:
    files.download(path)
    print(f'  Downloaded: {os.path.basename(path)}')

print('\nDone! Use model_ptq_int8.tflite for Arduino deployment.')